In [87]:
import numpy as np
import pandas as pd
import os
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

data = df[['Price']].values

# ======================
# SPLIT
# ======================
split_raw = int(len(data) * 0.8)

train_data = data[:split_raw]
test_data = data[split_raw - 60:]

# ======================
# SCALE
# ======================
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

# ======================
# SEQUENCES
# ======================
def create_sequences(data, time_step=60):
    X, y = [], []
    for i in range(time_step, len(data)):
        X.append(data[i-time_step:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

time_step = 60

X_train, y_train = create_sequences(train_scaled, time_step)
X_test, y_test = create_sequences(test_scaled, time_step)

X_train = X_train.reshape(X_train.shape[0], time_step, 1)
X_test = X_test.reshape(X_test.shape[0], time_step, 1)

# ======================
# LSTM MODEL (TRAIN FIRST)
# ======================
lstm = Sequential()
lstm.add(Input(shape=(time_step, 1)))
lstm.add(LSTM(64, return_sequences=True))
lstm.add(LSTM(32))
lstm.add(Dense(1))

lstm.compile(optimizer='adam', loss='mse')

lstm.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    verbose=1
)

# ======================
# LSTM PREDICTION
# ======================
lstm_pred = lstm.predict(X_test)
lstm_pred = scaler.inverse_transform(lstm_pred).flatten()

y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

# ======================
# BUILD HYBRID DATASET
# ======================
hybrid_df = df.iloc[split_raw:].copy().reset_index(drop=True)

hybrid_df = hybrid_df.iloc[:len(lstm_pred)].copy()

hybrid_df['LSTM_Pred'] = lstm_pred
hybrid_df['Lag1'] = hybrid_df['Price'].shift(1)

hybrid_df = hybrid_df.dropna().reset_index(drop=True)

# ======================
# TARGET
# ======================
X = hybrid_df[['Lag1', 'LSTM_Pred']]
y = hybrid_df['Price'] - hybrid_df['Lag1']

split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

actual = hybrid_df['Price'].iloc[split:].values

# ======================
# RANDOM FOREST
# ======================
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

# ======================
# PREDICTION
# ======================
pred_diff = rf.predict(X_test)
final_pred = X_test['Lag1'].values + pred_diff

# ======================
# EVALUATION
# ======================
rmse = np.sqrt(mean_squared_error(actual, final_pred))
mae = mean_absolute_error(actual, final_pred)
r2 = r2_score(actual, final_pred)
mape = np.mean(np.abs((actual - final_pred) / actual)) * 100

print("\n=== HYBRID MODEL RESULTS (FIXED) ===")
print(f"MAE: {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2: {r2:.6f}")
print(f"MAPE: {mape:.6f}")

# ======================
# SAVE MODEL
# ======================
os.makedirs("models", exist_ok=True)
joblib.dump(rf, "models/hybrid_rf.pkl")
lstm.save("models/hybrid_lstm.keras")

Epoch 1/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 8s 37ms/step - loss: 0.0146
Epoch 2/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 4.9611e-04
Epoch 3/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 4.9988e-04
Epoch 4/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 4.6220e-04
Epoch 5/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 4.6142e-04
Epoch 6/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 4.3537e-04
Epoch 7/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 4.4619e-04
Epoch 8/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 4.5960e-04
Epoch 9/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 4.2926e-04
Epoch 10/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - loss: 3.7848e-04
Epoch 11/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 3.9847e-04
Epoch 12/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 3.6154e-04
Epoch 13/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 3.4664e-04
Epoch 14/20
76/76 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 3.3598e-04
Epoch 15/20
76/76 ━